# Notebook #2: Isolation Forest Anomaly Detection

This notebook develops and evaluates an Isolation Forest (IF) for detecting abnormal engine behavior in the N-CMAPSS DS08a dataset. The model is trained on **healthy flights only** (`hs == 1`) and produces a continuous anomaly score for every flight, where higher values mean more abnormal. Health status is withheld from training and used afterward for evaluation.

The notebook starts from a **baseline** that uses the 12 physical sensors selected in Notebook #1. The baseline separates healthy from unhealthy flights only weakly (validation ROC-AUC ≈ 0.57), and its performance differs sharply by flight class. Both observations point to the same problem: **sensor readings are driven mostly by how the plane is flying (altitude, speed, throttle, inlet temperature), not by how healthy the engine is.**

The rest of the notebook tests whether removing that flight-condition effect lets the IF see degradation more clearly, then examines the final model's behavior in detail. Hyperparameter tuning is covered separately in Notebook #3.

## Outline:
1. Setup
2. Load Data
3. Train / Validation / Test Split
4. Helper Functions
5. Experiment A: Baseline and Downsampling Check
6. Experiment B: Operating-Condition Correction
7. Experiment C: Smoothing Scores Across Flights
8. Experiment D: Adding Virtual Sensors
9. Results Summary & Model Selection
10. Why the Correction Works (T48 Example)
11. Performance by Engine and Flight Class
12. Score Distributions: Healthy vs. Unhealthy
13. Score Trajectories Across Flight Cycles
14. Anomaly Score and Degradation Severity
15. Alarm Threshold, Detection Delay, & False Alarms
16. Summary for the Report

## 1.0 Setup

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "4"   # limit CPU threads for the gradient-boosting models

### 1.1 Import Libraries

* `train_test_split` creates the engine-level split (Section #3)
* `HistGradientBoostingRegressor` is used to predict what each sensor *should* read for a healthy engine under given flight conditions (Section #6)
* `roc_curve`, `precision_score`, `recall_score`, and `fbeta_score` are used for the threshold-based metrics (Section #15)
* `plotly.express` is used for the figures in Sections #11–15

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest, HistGradientBoostingRegressor
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_score, recall_score, fbeta_score
)

## 2.0 Load Data

Two rebuilt files are used (both created by `build_ds08a_from_h5.py` from NASA's original HDF5 file):

| File | What It Contains | Rows |
|:----:|------------------|:----:|
| `ds08a_rows.parquet` | Downsampled time points (1 per 76s for FC1; 127s for FC2; 180s for FC3), same data as Notebook #1 | 67,328 |
| `ds08a_flights.parquet` | 1 row per flight, with summary statistics computed from the **full 1 Hz** data | 994 |

The second file is used only in Section #5 to check whether downsampling lost information.

In [4]:
DATA_DIR = "../data"

rows = pd.read_parquet(f"{DATA_DIR}/ds08a_rows.parquet")
flights_1hz = pd.read_parquet(f"{DATA_DIR}/ds08a_flights.parquet")

print("Downsampled rows:", rows.shape)
print("1 Hz flight summaries:", flights_1hz.shape)

Downsampled rows: (67328, 50)
1 Hz flight summaries: (994, 182)


### 2.1 Define Groups of Columns

These follow Notebook #1, Sections #2.2 & 3.3:
* **Operating Conditions** describe how the plane is flying. They are *not* model inputs, but they are used to correct the sensors in Section #6.
* **Baseline Sensors** are the 12 physical sensors selected in Notebook #1 (`P21` and `P40` dropped as redundant).
* **Virtual Sensors** are quantities the simulator *calculates* (internal flows, stall margins). They are not physical measurements, so they are tested separately in Section #8.
* Identifiers, health labels, degradation parameters (`*_mod`), and `RUL` are never used as model inputs.

In [5]:
operating_cols = ["alt", "Mach", "TRA", "T2"]

baseline_sensor_cols = [
    "T24", "T30", "T48", "T50",
    "P15", "P2", "P24", "Ps30", "P50",
    "Nf", "Nc", "Wf"
]

virtual_cols = [
    "T40", "P30", "P45",
    "W21", "W22", "W25", "W31", "W32", "W48", "W50",
    "SmFan", "SmLPC", "SmHPC", "phi"
]

## 3.0 Train / Validation / Test Split

To prevent information leakage, the data are split by **engine** (`unit`) rather than by individual flight. Flights from the same engine share engine-specific characteristics and are related over time, so letting one engine appear in several partitions may make results look better than they really are.

The 15 engines are divided into 9 training, 3 validation, and 3 test engines, stratified by flight class so that FC1, FC2, and FC3 appear in every partition. This is the same split used by the rest of the team.

```text
9 training engines
       ├── hs = 1 ─────> FIT the correction models AND the Isolation Forest
       └── hs = 0 ─────> Not used

3 validation engines
       └── all flights ─> Compare Experiments / Choose  Final Model

3 test engines
       └── all flights ─> Final Evaluation Only
```

In [6]:
engine_df = rows.groupby("unit").agg(Fc=("Fc", "first")).reset_index()

train_engines, temp_engines = train_test_split(
    engine_df, test_size=0.40, stratify=engine_df["Fc"], random_state=42
)
val_engines, test_engines = train_test_split(
    temp_engines, test_size=0.50, stratify=temp_engines["Fc"], random_state=42
)

train_units = sorted(train_engines["unit"].tolist())
val_units   = sorted(val_engines["unit"].tolist())
test_units  = sorted(test_engines["unit"].tolist())

print("Training engines:  ", train_units)
print("Validation engines:", val_units)
print("Test engines:      ", test_units)

Training engines:   [2, 3, 5, 6, 7, 8, 9, 13, 15]
Validation engines: [4, 10, 14]
Test engines:       [1, 11, 12]


In [7]:
# Flight class representation in each partition
engine_fc = engine_df.set_index("unit")["Fc"]
for name, units in [("Train", train_units), ("Validation", val_units), ("Test", test_units)]:
    print(f"{name:<11}", engine_fc.loc[units].value_counts().sort_index().to_dict())

Train       {1: 3, 2: 3, 3: 3}
Validation  {1: 1, 2: 1, 3: 1}
Test        {1: 1, 2: 1, 3: 1}


## 4.0 Helper Functions

3 small functions are reused throughout so that **only one thing changes per experiment**.

### 4.1 `flight_means()`
Collapses each flight's time points into one row by taking the mean of each column.

In [8]:
def flight_means(row_df, cols):
    """One row per (unit, cycle): mean of each column, plus hs and Fc."""
    return (
        row_df.groupby(["unit", "cycle"])
              .agg(**{c: (c, "mean") for c in cols},
                   hs=("hs", "first"),
                   Fc=("Fc", "first"))
              .reset_index()
    )

### 4.2 `residualize()`: the operating-condition correction

This is the key step of the notebook. For each sensor:

1. Take **healthy rows (`hs == 1`) from the training engines only**.
2. Fit a model that predicts the sensor from the four operating conditions, e.g. `T48 ≈ f(alt, Mach, TRA, T2)`. This model learns *what a healthy engine's T48 should be under each flight condition*.
3. Apply that model to **every** row (all engines, healthy or not) and compute:

$$\text{residual} = \text{actual sensor value} - \text{predicted healthy value}$$

In plain language, the residual answers: *"Given how the plane was flying at that moment, how far off is this engine from what a healthy engine would read?"* In other words, a healthy engine should have residuals near 0, while a degrading engine should drift away from 0.

#### Why Gradient Boosting?

The relationship between flight conditions and sensors is curved, not a straight line. A linear regression version was also tried during development and reached only ~0.65 ROC-AUC.

#### Leakage Check

The correction models only ever see healthy data from training engines, so nothing from validation or test engines influences them. `hs` is used only to choose that training data, which is the same rule already used for the Isolation Forest itself.

In [9]:
def residualize(row_df, cols):
    """Replace each sensor with (actual - predicted healthy value given operating conditions).
    Correction models are fit on HEALTHY rows from TRAINING engines only."""
    train_rows = row_df[row_df["unit"].isin(train_units) & (row_df["hs"] == 1)]

    resid = row_df[["unit", "cycle", "hs", "Fc"]].copy()
    r2 = {}

    for c in cols:
        model = HistGradientBoostingRegressor(max_iter=200, random_state=0)
        model.fit(train_rows[operating_cols], train_rows[c])

        resid[c] = row_df[c] - model.predict(row_df[operating_cols])
        r2[c] = model.score(train_rows[operating_cols], train_rows[c])   # how well flight conditions explain the sensor

    return resid, pd.Series(r2, name="R2_train_healthy")

### 4.3 `evaluate_iforest()`

Fits an Isolation Forest on healthy training flights, scores every flight (negating `score_samples()` so that higher = more anomalous), and reports ROC-AUC and PR-AUC on the validation and test engines.

Hyperparameters are kept fixed (`n_estimators=200`, `max_samples="auto"`, `max_features=1.0`) so that any change in performance comes from the **features**, not the settings. Notebook #3 shows that these settings have little effect once the features are corrected.

The optional `smooth` argument is explained in Section #7.

In [10]:
def evaluate_iforest(flight_df, feature_cols, label, smooth=0):
    """Fit IF on healthy TRAINING flights; report validation and test ROC-AUC / PR-AUC.
    hs = 1 -> healthy, hs = 0 -> unhealthy (positive class)."""
    train_healthy = flight_df[
        flight_df["unit"].isin(train_units) & (flight_df["hs"] == 1)
    ]

    model = IsolationForest(
        n_estimators=200,
        max_samples="auto",
        max_features=1.0,
        random_state=42
    ).fit(train_healthy[feature_cols])

    scored = flight_df.sort_values(["unit", "cycle"]).copy()
    scored["anomaly_score"] = -model.score_samples(scored[feature_cols])

    if smooth:
        # median of the current flight and the previous (smooth - 1) flights, within each engine
        scored["anomaly_score"] = (
            scored.groupby("unit")["anomaly_score"]
                  .transform(lambda s: s.rolling(smooth, min_periods=1).median())
        )

    out = {"model": label}
    for name, units in [("val", val_units), ("test", test_units)]:
        part = scored[scored["unit"].isin(units)]
        y = 1 - part["hs"]                       # unhealthy = 1
        out[f"{name}_ROC_AUC"] = roc_auc_score(y, part["anomaly_score"])
        out[f"{name}_PR_AUC"] = average_precision_score(y, part["anomaly_score"])

    return out, scored

## 5.0 Experiment A: Baseline and Downsampling Check

### 5.1 Baseline
The baseline fits the Isolation Forest on the **raw per-flight means** of the 12 baseline sensors. Operating conditions are left out of the features (Notebook #1, Section 3.0), because unusual but valid flight conditions would otherwise look anomalous.

### 5.2 Does downsampling lose information?
The team downsampled the original 1 Hz data (~8.6M rows) to ~67K rows to fit within available compute. To check whether this threw away useful signal, the same baseline is fit a second time using flight means computed from **every second** of the original 1 Hz data. If the results are similar, downsampling did not cost us information, and adding more rows would not help.

In [11]:
results = []

# 1. Baseline: raw sensor means from downsampled rows
res, base_scores = evaluate_iforest(
    flight_means(rows, baseline_sensor_cols),
    baseline_sensor_cols,
    "1. Baseline: raw means, downsampled"
)
results.append(res)

# 2. Same model, but flight means from the full 1 Hz data
f1 = flights_1hz.rename(columns={f"{c}_mean": c for c in baseline_sensor_cols})
res, _ = evaluate_iforest(f1, baseline_sensor_cols, "2. Raw means, full 1 Hz")
results.append(res)

pd.DataFrame(results).round(3)

,model,val_ROC_AUC,val_PR_AUC,test_ROC_AUC,test_PR_AUC
0,"1. Baseline: raw means, downsampled",0.572,0.729,0.641,0.719
1,"2. Raw means, full 1 Hz",0.550,0.716,0.633,0.718


### 5.3 Experiment A: Result

* The baseline separates healthy from unhealthy flights only weakly: validation ROC-AUC $\approx 0.572$, with a PR-AUC ($\approx 0.729$) barely above the PR baseline (the share of unhealthy flights, $\approx 0.69$ in validation).
* Using the full 1 Hz data gives essentially the same performance. Averaging over a flight already smooths out second-to-second detail, and the number of flights (994) and engines (15) does not change. **Downsampling did not cost us information, so the downsampled data is kept.**

## 6.0 Experiment B: Operating-Condition Correction

### 6.1 How much do flight conditions explain each sensor?

Before using the residuals, it is worth checking how strongly the four operating conditions drive each sensor. The $R^2$ below is the share of each sensor's variation (on healthy training rows) that the correction model explains from flight conditions alone. An $R^2$ of 0.999 means 99.9%.

In [12]:
resid_12, r2_12 = residualize(rows, baseline_sensor_cols)

print("Share of each sensor explained by operating conditions (healthy training rows):")
r2_12.round(4)

Share of each sensor explained by operating conditions (healthy training rows):


T24     0.9996
T30     0.9996
T48     0.9996
T50     0.9991
P15     0.9998
P2      1.0000
P24     0.9998
Ps30    0.9997
P50     0.9999
Nf      0.9999
Nc      0.9996
Wf      0.9997
Name: R2_train_healthy, dtype: float64

Every sensor is explained almost entirely (>99.9%) by flight conditions. This is why the baseline struggled: almost all of the variation the IF saw was about *how the plane was flying*. The small part left over (i.e., the *residual*) is where engine health shows up.

### 6.2 Fit the Isolation Forest on residuals

Same 12 sensors and same IF settings as the baseline. The only change is that each sensor is replaced by its **residual** before taking flight means.

In [13]:
res, _ = evaluate_iforest(
    flight_means(resid_12, baseline_sensor_cols),
    baseline_sensor_cols,
    "3. Residuals, 12 sensors"
)
results.append(res)
pd.DataFrame(results).round(3)

,model,val_ROC_AUC,val_PR_AUC,test_ROC_AUC,test_PR_AUC
0,"1. Baseline: raw means, downsampled",0.572,0.729,0.641,0.719
1,"2. Raw means, full 1 Hz",0.550,0.716,0.633,0.718
2,"3. Residuals, 12 sensors",0.848,0.872,0.815,0.906


### 6.3 Experiment B: Result

Validation ROC-AUC jumps from about **0.57 to 0.85**, with no change to the model or its hyperparameters. This is the single largest improvement in the notebook, and it confirms that flight conditions were masking the degradation signal.

## 7.0 Experiment C: Smoothing Scores Across Flights

Engine degradation builds up **gradually** over many flights, but the score for any single flight can jump around by chance (e.g., an unusual flight profile).

Instead of judging each flight on its own, each flight's score is replaced by the **median of that flight and the 4 flights before it**. This:
* Only looks backward, so it could be used in real time (no peeking at future flights), and
* Stops a single noisy flight from triggering a false alarm or hiding a real problem.

Thus, conceptually, for one engine:

```text
cycle:          1     2     3     4     5     6    ...
raw score:     0.45  0.47  0.62  0.46  0.48  0.49  ...
smoothed:      0.45  0.46  0.47  0.47  0.47  0.48  ...   <-- the 0.62 spike no longer dominates
```

The window length (5 flights) is examined in the sensitivity analysis later on in Notebook #3.

In [14]:
res, _ = evaluate_iforest(
    flight_means(resid_12, baseline_sensor_cols),
    baseline_sensor_cols,
    "4. Residuals, 12 sensors + 5-flight median",
    smooth=5
)
results.append(res)
pd.DataFrame(results).round(3)

,model,val_ROC_AUC,val_PR_AUC,test_ROC_AUC,test_PR_AUC
0,"1. Baseline: raw means, downsampled",0.572,0.729,0.641,0.719
1,"2. Raw means, full 1 Hz",0.550,0.716,0.633,0.718
2,"3. Residuals, 12 sensors",0.848,0.872,0.815,0.906
3,"4. Residuals, 12 sensors + 5-flight median",0.897,0.946,0.801,0.907


### 7.1 Experiment C: Result

Smoothing improves validation ROC-AUC from about 0.85 to 0.90. Once flight-to-flight noise is reduced, the gradual upward trend in scores during degradation is easier to separate.

## 8.0 Experiment D: Adding Virtual Sensors

The dataset includes 14 virtual sensors, which are quantities the simulator calculates for parts of the engine that the physical sensors don't directly measure (e.g., internal airflows `W21`–`W50` and stall margins `SmFan`, `SmLPC`, `SmHPC`).

They are corrected for operating conditions in exactly the same way, then added alongside the 12 physical sensors.

**Caveat:** Virtual sensors are simulator outputs, not physical measurements, and would not be available on a real engine in the same form. Results are therefore reported both with them (Models #5–6) and without them (Models #3–4), so the contribution of physical sensors alone can be assessed.

In [15]:
all_cols = baseline_sensor_cols + virtual_cols
resid_all, _ = residualize(rows, all_cols)

# 5. Virtual sensors, no smoothing
res, _ = evaluate_iforest(
    flight_means(resid_all, all_cols),
    all_cols,
    "5. Residuals, 12 sensors + virtual"
)
results.append(res)

# 6. Virtual sensors + smoothing
res, best_scores = evaluate_iforest(
    flight_means(resid_all, all_cols),
    all_cols,
    "6. Residuals, 12 sensors + virtual + 5-flight median",
    smooth=5
)
results.append(res)

## 9.0 Results Summary & Model Selection

All six experiments side by side. Each row changes one thing from the row it builds on:

| # | What Changed |
|:-:|--------------|
| 1 | Baseline: raw sensor means |
| 2 | Same, but 1 Hz data instead of downsampled |
| 3 | Operating-condition correction (residuals) |
| 4 | #3 + 5-flight rolling median |
| 5 | #3 + virtual sensors |
| 6 | #3 + virtual sensors + 5-flight rolling median |

In [16]:
results_df = pd.DataFrame(results).round(3)
results_df

,model,val_ROC_AUC,val_PR_AUC,test_ROC_AUC,test_PR_AUC
0,"1. Baseline: raw means, downsampled",0.572,0.729,0.641,0.719
1,"2. Raw means, full 1 Hz",0.550,0.716,0.633,0.718
2,"3. Residuals, 12 sensors",0.848,0.872,0.815,0.906
3,"4. Residuals, 12 sensors + 5-flight median",0.897,0.946,0.801,0.907
4,"5. Residuals, 12 sensors + virtual",0.893,0.911,0.842,0.929
5,"6. Residuals, 12 sensors + virtual + 5-flight ...",0.923,0.965,0.832,0.928


### 9.1 Select the Final Model

**The model is chosen using validation results only.** The test columns are shown for completeness, but the test set was *NOT* used to make choices; otherwise the test score would no longer be an honest estimate of generalizable performance on new engines.

In [17]:
best = results_df.loc[results_df["val_ROC_AUC"].idxmax()]
best

model           6. Residuals, 12 sensors + virtual + 5-flight ...
val_ROC_AUC                                                 0.923
val_PR_AUC                                                  0.965
test_ROC_AUC                                                0.832
test_PR_AUC                                                 0.928
Name: 5, dtype: object

### 9.2 Interpretation

* **Selected Model: #6** (i.e., residuals + virtual sensors + 5-flight median), with the highest validation ROC-AUC ($0.92$).
* **Test Result (Single Seed): $\text{ROC-AUC} \approx 0.83, \text{PR-AUC} \approx 0.93$ (PR baseline = share of unhealthy flights $\approx 0.65$). Notebook #3 re-evaluates this pipeline averaged over 5 random seeds, which is the more reliable number to report.
* Row #5 has a slightly higher *test* ROC-AUC than #6, but it is **NOT** selected, because choosing by test score would leak test information into model selection.
* The gap between validation ($0.92$) and test ($0.83$) is a reminder that with only 3 engines per partition, results can vary noticeably depending on which engines are held out.
* If the report uses **physical sensors only**, Model #3 is the comparable choice ($\text{validation}=0.85, \text{test}=0.82$).

Sections #10–15 will compare the **Baseline (#1)** with the **Final Model (#6)** in more detail.

## 10.0 Why the Correction Works (`T48` Example)

`T48` is the gas temperature between the high- and low-pressure turbines, measured in degrees Rankine (°R). Rankine is an absolute scale with Fahrenheit-sized degrees, so a **difference** of $1\degree R$ equals $1\degree F$ ($\approx 0.56 \degree C$).

Using the validation engines, the table below compares three quantities:

1. **Raw flight-mean SD:** How much the average `T48` varies from one flight to the next *before* correction (mostly due to different flight conditions)
2. **Degradation Shift:** How much higher the corrected `T48` is, on average, in unhealthy flights compared with healthy flights
3. **Residual Flight-Mean SD (Healthy Flights):** How much healthy flights still vary *after* correction

If the degradation shift is small compared with (1) but large compared with (3), the correction is what makes degradation visible.

In [18]:
val_rows  = rows[rows["unit"].isin(val_units)]
val_resid = resid_12[resid_12["unit"].isin(val_units)]

raw_flight = val_rows.groupby(["unit", "cycle"])["T48"].mean()
res_flight = val_resid[val_resid["hs"] == 1].groupby(["unit", "cycle"])["T48"].mean()
shift = (val_resid.loc[val_resid["hs"] == 0, "T48"].mean()
         - val_resid.loc[val_resid["hs"] == 1, "T48"].mean())

t48 = pd.DataFrame({"°R": {
    "Raw flight-mean SD (flight-to-flight noise)": raw_flight.std(),
    "Degradation shift (unhealthy - healthy residual)": shift,
    "Residual flight-mean SD, healthy flights": res_flight.std(),
}})
t48["≈ °C (difference)"] = t48["°R"] * 5 / 9
t48.round(2)

,°R,≈ °C (difference)
Raw flight-mean SD (flight-to-flight noise),17.66,9.81
Degradation shift (unhealthy - healthy residual),9.56,5.31
"Residual flight-mean SD, healthy flights",2.04,1.13


### 10.1 Interpretation

Before correction, a degradation shift of roughly $10 \degree R$ ($\approx 5 \degree C$) is hidden inside roughly $18 \degree R$ ($\approx 10 \degree C$) of flight-to-flight variation, so the IF cannot tell an unhealthy engine apart from an unusual flight. After correction, healthy flights vary by only about $2 \degree R$ ($\approx 1 \degree C$), so the same shift stands out clearly (about 5 times larger than the remaining noise).

A helpful analogy: checking for a fever by comparing someone's temperature to *their normal temperature for what they were doing* (resting, exercising), rather than to one fixed number.

## 11.0 Performance by Engine & Flight Class

Pooled metrics can hide engines where the method struggles. Because each validation and test partition contains *exactly one* engine per flight class, the per-engine results below also serve as a flight-class breakdown.

To prepare the comparisons in Sections #11–15, the baseline and final-model scores are combined into one table for the 6 held-out engines (i.e., validation + test).

In [19]:
heldout_units = val_units + test_units

compare_df = pd.concat([
    base_scores.assign(model="Baseline (#1)"),
    best_scores.assign(model="Final (#6)"),
])
compare_df = compare_df[compare_df["unit"].isin(heldout_units)].copy()

compare_df["partition"] = np.where(compare_df["unit"].isin(val_units), "Validation", "Test")
compare_df["health_status"] = compare_df["hs"].map({1: "Healthy", 0: "Unhealthy"})
compare_df["engine"] = ("Engine " + compare_df["unit"].astype(int).astype(str)
                        + " (FC" + compare_df["Fc"].astype(int).astype(str) + ")")

In [20]:
per_engine = (
    compare_df.groupby(["partition", "unit", "Fc", "model"])
              .apply(lambda d: roc_auc_score(1 - d["hs"], d["anomaly_score"]))
              .unstack("model")
              .round(3)
)
per_engine

model               Baseline (#1)  Final (#6)
partition  unit Fc                           
Test       1    1           0.829       0.856
           11   2           0.493       0.826
           12   3           0.524       0.810
Validation 4    2           0.499       0.918
           10   1           0.740       1.000
           14   3           0.368       0.832

### 11.1 Interpretation

* With the baseline, performance depended heavily on flight class. The FC3 validation engine scored below $0.5$, meaning the baseline ranked its *healthy* flights as more anomalous than its unhealthy ones. Long-haul flights reach conditions that look "unusual" to a model that cannot account for flight profile.
* With the final model, every held-out engine scores well above $0.5$, across all three flight classes. Because each sensor is now compared with what is normal for *that* flight profile, flight class no longer drives the score.

### 11.2 Do Scores Still Depend on Operating Conditions?

Notebook #1 (Section #3.0) noted that performance would be checked across operating conditions. If the correction worked, a **healthy** flight's anomaly score should have little relationship with how the plane was flying.

The check below uses only **healthy held-out flights** and computes the Pearson correlation between each flight's anomaly score and its mean altitude, Mach number, throttle angle (`TRA`), and inlet temperature (`T2`). Values near $0$ mean the score is not tracking flight conditions.

In [21]:
flight_conditions = rows.groupby(["unit", "cycle"])[operating_cols].mean().reset_index()

healthy_heldout = (
    compare_df[compare_df["hs"] == 1]
        .merge(flight_conditions, on=["unit", "cycle"], how="left")
)

oc_correlation = (
    healthy_heldout.groupby("model")
                   .apply(lambda d: pd.Series({c: d["anomaly_score"].corr(d[c]) for c in operating_cols}))
                   .round(3)
)
oc_correlation

,alt,Mach,TRA,T2
model,,,,
Baseline (#1),0.532,0.314,0.46,-0.602
Final (#6),0.281,0.250,0.33,-0.282


#### 11.2.1 Interpretation

* **Baseline:** Healthy-flight scores track flight conditions closely ($|r| \approx 0.31$ to $0.60$). Flights at higher altitude, higher throttle, or colder inlet temperature were scored as more anomalous, even though the engine was healthy.
* **Final Model:** The same correlations drop to $|r| \approx 0.25$ to $0.33$, roughly half of the baseline's. The correction removes much of the flight-condition effect, but *NOT* all of it.
* Part of the remaining correlation may come from the engines themselves rather than from flight conditions directly: each engine flies only one flight class, so an engine whose healthy scores run slightly high will also appear to "prefer" that class's altitudes and throttle settings.

This addresses the check proposed in Notebook #1: operating context still has a small effect on the scores, but it is no longer the main driver.

## 12.0 Score Distributions: Healthy vs. Unhealthy

**Caption for Visualization Below:** Anomaly scores for every held-out flight, split by health status, for the baseline and the final model. A useful anomaly score should place unhealthy flights clearly higher than healthy ones.

In [22]:
fig = px.box(
    compare_df,
    x="health_status",
    y="anomaly_score",
    color="health_status",
    facet_col="model",
    points="all",
    hover_data=["unit", "cycle", "Fc", "partition"],
    category_orders={"model": ["Baseline (#1)", "Final (#6)"], "health_status": ["Healthy", "Unhealthy"]},
    title="Anomaly Scores by Health Status (Held-Out Engines)",
    labels={"health_status": "Health Status", "anomaly_score": "Anomaly Score"},
)
fig.update_layout(showlegend=False)
fig.show()

In [23]:
# Numeric summary of the same figure
(compare_df.groupby(["model", "health_status"])["anomaly_score"]
           .describe()[["count", "25%", "50%", "75%"]]
           .round(3))

count    25%    50%    75%
model         health_status                            
Baseline (#1) Healthy        132.0  0.437  0.465  0.504
              Unhealthy      266.0  0.452  0.488  0.523
Final (#6)    Healthy        132.0  0.491  0.512  0.545
              Unhealthy      266.0  0.567  0.667  0.699

### 12.1 Interpretation

* For the baseline, the healthy and unhealthy boxes overlap heavily, which matches its ROC-AUC near $0.6$.
* For the final model, the unhealthy flights sit clearly higher: the median unhealthy score is *above* the upper quartile of healthy scores.
* Some overlap remains, mostly from unhealthy flights just after the transition, when degradation has only begun (*see Section #13*).

## 13.0 Score Trajectories Across Flight Cycles

**Caption for Visualization Below:** Each held-out engine's anomaly score over its life, colored by health status. The point where the color changes is the transition from healthy to unhealthy. A good anomaly score should stay low and flat while the engine is healthy, then rise as degradation progresses.

In [24]:
fig = px.line(
    compare_df,
    x="cycle",
    y="anomaly_score",
    color="health_status",
    line_dash="model",
    facet_col="engine",
    facet_col_wrap=3,
    markers=False,
    category_orders={"model": ["Baseline (#1)", "Final (#6)"], "health_status": ["Healthy", "Unhealthy"]},
    hover_data=["partition"],
    title="Anomaly Score Across Flight Cycles: Baseline (Dashed) vs. Final (Solid)",
    labels={"cycle": "Flight cycle", "anomaly_score": "Anomaly Score", "health_status": "Health Status"},
)
fig.update_yaxes(matches="y")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=650)
fig.show()

In [25]:
# Final Model Only, which will be easier to read for the report
fig = px.line(
    compare_df[compare_df["model"] == "Final (#6)"],
    x="cycle",
    y="anomaly_score",
    color="health_status",
    facet_col="engine",
    facet_col_wrap=3,
    markers=True,
    category_orders={"health_status": ["Healthy", "Unhealthy"]},
    hover_data=["partition"],
    title="Final Model: Anomaly Score Across Flight Cycles (Held-Out Engines)",
    labels={"cycle": "Flight cycle", "anomaly_score": "Anomaly Score", "health_status": "Health Status"},
)
fig.update_yaxes(matches="y")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=650)
fig.show()

### 13.1 Interpretation

The final model's scores stay relatively flat during healthy operation and climb steadily after the transition, which is the "degradation trajectory" behavior the project set out to test. The baseline's scores move up and down from flight to flight with no clear relationship to health status.

Two limitations are visible:
* **Scores rise gradually** after the transition rather than jumping, so the first few unhealthy flights are often scored similarly to healthy ones. This was the main source of missed detections.
* A few healthy flights still score relatively high, which would produce false alarms at a low alarm threshold.

## 14.0 Anomaly Score & Degradation Severity

If the anomaly score reflects how far an engine has drifted from healthy behavior, it should increase as the engine moves through its unhealthy period.

For each unhealthy flight, **unhealthy progress** is defined as its position within that engine's unhealthy period, from `0` (first unhealthy flight) to `1` (last flight before failure). The correlation between progress and anomaly score is then computed for the baseline and the final model.

In [26]:
unhealthy = compare_df[compare_df["hs"] == 0].copy()

first_c = unhealthy.groupby(["model", "unit"])["cycle"].transform("min")
last_c  = unhealthy.groupby(["model", "unit"])["cycle"].transform("max")
unhealthy["unhealthy_progress"] = (unhealthy["cycle"] - first_c) / (last_c - first_c)

pooled_r = unhealthy.groupby("model")[["unhealthy_progress", "anomaly_score"]].apply(
    lambda d: d["unhealthy_progress"].corr(d["anomaly_score"])
)
print("Pooled Pearson correlation (held-out engines):")
pooled_r.round(3)

Pooled Pearson correlation (held-out engines):


model
Baseline (#1)    0.304
Final (#6)       0.883
dtype: float64

In [27]:
per_engine_r = (
    unhealthy.groupby(["unit", "Fc", "model"])
             .apply(lambda d: d["unhealthy_progress"].corr(d["anomaly_score"]))
             .unstack("model")
             .round(3)
)
per_engine_r

,model,Baseline (#1),Final (#6)
unit,Fc,,
1,1,0.274,0.982
4,2,0.620,0.925
10,1,0.277,0.951
11,2,0.373,0.940
12,3,-0.088,0.869
14,3,0.410,0.862


In [28]:
fig = px.scatter(
    unhealthy,
    x="unhealthy_progress",
    y="anomaly_score",
    color="engine",
    facet_col="model",
    trendline="ols",
    category_orders={"model": ["Baseline (#1)", "Final (#6)"]},
    title="Anomaly Score Across the Unhealthy Period (Held-Out Engines)",
    labels={"unhealthy_progress": "Progress Through Unhealthy Period",
            "anomaly_score": "Anomaly Score", "engine": "Engine"},
)
fig.show()

### 14.1 Interpretation

The baseline showed only a weak link between score and degradation progress (pooled $r \approx 0.30$, and one engine was slightly negative). With the final model, the link is strong and consistent across every held-out engine (pooled $r \approx 0.88$; per engine $0.86$ to $0.98$): anomaly scores rise steadily as engines approach failure. This supports interpreting the score as a **continuous measure of departure from healthy behavior**, not just a healthy/unhealthy flag.

Note that "unhealthy progress" is based on flight position, not on a measured amount of damage. A natural extension is to compare scores directly against the simulator's degradation parameters (`*_mod`), which are available in the rebuilt data but were not used as model inputs.

## 15.0 Alarm Threshold, Detection Delay, & False Alarms

ROC-AUC and PR-AUC measure how well the anomaly scores **rank** unhealthy flights above healthy ones, across *every* possible alarm threshold. This makes them the right metrics for comparing models (Sections #5–9), but they do not show what happens once an actual alarm threshold is set.

Because missing a degrading engine is more costly than an extra inspection, this project is **recall-focused**: the goal is to catch most unhealthy flights while accepting some false alarms. This section therefore picks a single alarm threshold and reports:

| Metric | Definition | Question It Answers |
|:------:|------------|---------------------|
| Recall | $\frac{TP}{TP + FN}$ | What share of unhealthy flights trigger an alarm? |
| False-Alarm Rate (FPR) | $\frac{FP}{FP + TN}$ | What share of healthy flights trigger an alarm? |
| Precision | $\frac{TP}{TP + FP}$ | When an alarm goes off, how often is the flight truly unhealthy? |
| $F_2$ | $\frac{5 \cdot \text{Precision} \cdot \text{Recall}}{4 \cdot \text{Precision} + \text{Recall}}$ | Single summary that weighs Recall **twice** as much as Precision |
| Detection Delay | First alarm cycle $-$ first unhealthy cycle | How many flights after degradation onset is the engine flagged? |

Here, a "positive" is an **unhealthy** flight (`hs == 0`), and an alarm is raised when a flight's (smoothed) anomaly score is at or above the threshold.

All results in this section use the **Final Model (#6)**.

### 15.1 Choose the Threshold on Validation Engines

The threshold is chosen using the **validation engines only**, then applied unchanged to the test engines (the same rule used for model selection in Section #9). The threshold is set to the **highest score that still catches at least 90% of unhealthy validation flights**.

In [29]:
def threshold_for_recall(y_true, scores, target_recall):
    """Highest threshold whose recall on (y_true, scores) is >= target_recall."""
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    return thresholds[np.argmax(tpr >= target_recall)]


def alarm_metrics(df, threshold):
    """Recall, false-alarm rate, precision, and F2 at a fixed threshold (unhealthy = positive)."""
    y_true = 1 - df["hs"]
    y_pred = (df["anomaly_score"] >= threshold).astype(int)
    return {
        "Recall": recall_score(y_true, y_pred),
        "FPR": ((y_pred == 1) & (y_true == 0)).sum() / (y_true == 0).sum(),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "F2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
    }


final_scores = compare_df[compare_df["model"] == "Final (#6)"]
val_final  = final_scores[final_scores["partition"] == "Validation"]
test_final = final_scores[final_scores["partition"] == "Test"]

TARGET_RECALL = 0.90
alarm_threshold = threshold_for_recall(1 - val_final["hs"], val_final["anomaly_score"], TARGET_RECALL)

print(f"Alarm threshold (chosen on validation, target recall = {TARGET_RECALL:.0%}): {alarm_threshold:.3f}")

Alarm threshold (chosen on validation, target recall = 90%): 0.560


### 15.2 Performance at the Threshold

In [30]:
threshold_results = pd.DataFrame(
    [alarm_metrics(val_final, alarm_threshold), alarm_metrics(test_final, alarm_threshold)],
    index=["Validation", "Test"]
)
threshold_results.round(3)

,Recall,FPR,Precision,F2
Validation,0.916,0.277,0.879,0.908
Test,0.691,0.015,0.988,0.735


### 15.3 Detection Delay & False Alarms by Engine

For each held-out engine:
* **Transition Cycle:** First flight labeled unhealthy (`hs == 0`)
* **First Alarm:** First flight *at or after* the transition with a score at or above the threshold
* **Detection Delay:** $\text{First Alarm} - \text{Transition Cycle}$ (in flights)
* **False Alarms:** Healthy flights (*before* the transition) with a score at or above the threshold
* **Missed Unhealthy Flights:** Unhealthy flights with a score below the threshold

In [31]:
def engine_alarm_summary(df, threshold):
    rows_out = []
    for unit, g in df.sort_values("cycle").groupby("unit"):
        alarm = g["anomaly_score"] >= threshold
        healthy, unhealthy = g["hs"] == 1, g["hs"] == 0
        transition = g.loc[unhealthy, "cycle"].min()
        first_alarm = g.loc[unhealthy & alarm, "cycle"].min()
        rows_out.append({
            "partition": g["partition"].iloc[0],
            "unit": unit,
            "Fc": int(g["Fc"].iloc[0]),
            "transition_cycle": transition,
            "first_alarm": first_alarm,
            "detection_delay": first_alarm - transition,
            "false_alarms": f"{(healthy & alarm).sum()} / {healthy.sum()}",
            "missed_unhealthy": f"{(unhealthy & ~alarm).sum()} / {unhealthy.sum()}",
        })
    return pd.DataFrame(rows_out).set_index(["partition", "unit"])


engine_alarms = engine_alarm_summary(final_scores, alarm_threshold)
engine_alarms

Fc  transition_cycle  first_alarm  detection_delay  \
partition  unit                                                       
Test       1      1                31           42               11   
Validation 4      2                22           27                5   
           10     1                30           32                2   
Test       11     2                21           24                3   
           12     3                18           19                1   
Validation 14     3                16           18                2   

                false_alarms missed_unhealthy  
partition  unit                                
Test       1          0 / 30          11 / 42  
Validation 4         10 / 21           5 / 50  
           10         0 / 29           2 / 49  
Test       11         1 / 20          15 / 40  
           12         0 / 17          12 / 41  
Validation 14         8 / 15           5 / 44

**Caption for Visualization Below:** Final-model anomaly scores for the 3 test engines, with the alarm threshold chosen on validation shown as a dotted line. Points above the line raise an alarm. The color change marks the transition from healthy to unhealthy.

In [32]:
fig = px.line(
    test_final,
    x="cycle",
    y="anomaly_score",
    color="health_status",
    facet_col="engine",
    markers=True,
    category_orders={"health_status": ["Healthy", "Unhealthy"]},
    title="Final Model: Test-Engine Scores vs. Alarm Threshold",
    labels={"cycle": "Flight Cycle", "anomaly_score": "Anomaly Score", "health_status": "Health Status"},
)
fig.add_hline(y=alarm_threshold, line_dash="dot", line_color="black",
              annotation_text="Alarm Threshold", annotation_position="top left")
fig.update_yaxes(matches="y")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

### 15.4 Sensitivity to the Recall Target

The 90% recall target is a design choice. To show the tradeoff between catching degradation and raising false alarms, the threshold is re-chosen on validation for several recall targets, and each threshold is then applied to the test engines.

In [33]:
tradeoff = []
for target in [0.70, 0.80, 0.90, 0.95, 0.99]:
    thr = threshold_for_recall(1 - val_final["hs"], val_final["anomaly_score"], target)
    test_metrics = alarm_metrics(test_final, thr)
    delays = engine_alarm_summary(test_final, thr)["detection_delay"]
    tradeoff.append({
        "target_recall (val)": target,
        "threshold": thr,
        **{f"test_{k}": v for k, v in test_metrics.items()},
        "test_mean_delay (flights)": delays.mean(),
    })

tradeoff_df = pd.DataFrame(tradeoff)
tradeoff_df.round(3)

,target_recall (val),threshold,test_Recall,test_FPR,test_Precision,test_F2,test_mean_delay (flights)
0,0.70,0.612,0.569,0.000,1.000,0.623,17.667
1,0.80,0.578,0.610,0.000,1.000,0.661,11.667
2,0.90,0.560,0.691,0.015,0.988,0.735,5.000
3,0.95,0.546,0.756,0.209,0.869,0.776,4.000
4,0.99,0.473,0.967,0.955,0.650,0.881,0.000


**Caption for Visualization Below:** Test-engine Recall and False-Alarm Rate (FPR) for each validation recall target. Moving right lowers the threshold, which catches more unhealthy flights at the cost of more false alarms.

In [34]:
plot_tradeoff = tradeoff_df.melt(
    id_vars="target_recall (val)",
    value_vars=["test_Recall", "test_FPR"],
    var_name="metric",
    value_name="value",
)
plot_tradeoff["metric"] = plot_tradeoff["metric"].map({"test_Recall": "Recall", "test_FPR": "False-Alarm Rate"})

fig = px.line(
    plot_tradeoff,
    x="target_recall (val)",
    y="value",
    color="metric",
    markers=True,
    title="Recall vs. False-Alarm Tradeoff on Test Engines",
    labels={"target_recall (val)": "Recall Target Used to Set Threshold (Validation)",
            "value": "Test Rate", "metric": "Metric"},
)
fig.update_yaxes(range=[0, 1.05])
fig.show()

### 15.5 Interpretation

* **At a 90% Recall Target:** The threshold chosen on validation ($\approx 0.560$) catches $91.6\%$ of unhealthy validation flights, but only $69.1\%$ of unhealthy **test** flights. The threshold does *NOT* fully carry over between engines.
* **The Direction of the Mismatch Differs by Engine:** The validation engines raised more false alarms (FPR $\approx 0.28$; Engines #4 & #14 flagged $10/21$ and $8/15$ healthy flights), while the test engines raised almost none (FPR $\approx 0.015$; $1$ false alarm across $67$ healthy flights) but missed more unhealthy flights. In other words, some engines simply run "hotter" or "cooler" than others, so a single cutoff is too strict for some and too loose for others.
* **Detection Delay:** Every held-out engine was eventually flagged, $1$ to $11$ flights after the transition (test engines: $11$, $3$, and $1$ flights). Every missed unhealthy test flight falls within the **first 40%** of its engine's unhealthy period; after that, all unhealthy flights are flagged. This matches the gradual rise in scores seen in Section #13.
* **Alarms Are Not Always Continuous:** Engines #11 & #12 were first flagged only $3$ and $1$ flights after their transitions, yet $15/40$ and $12/41$ of their unhealthy flights were still missed, because their scores dip back below the threshold a few times before staying above it.

#### 15.5.1 Tradeoffs

* **Recall vs. False Alarms:** Lowering the threshold catches degradation earlier and more completely, but false alarms rise sharply near the high-recall end. Moving the validation target from $90\%$ to $95\%$ raises test Recall from $0.69$ to $0.76$ and cuts the mean detection delay from $5$ to $4$ flights, but the false-alarm rate jumps from $0.015$ to $0.21$. At a $99\%$ target, Recall reaches $0.97$, but $95.5\%$ of healthy flights also trigger an alarm, which makes the alarm meaningless.
* **$F_2$ Favors the Lower Thresholds:** Because $F_2$ weighs Recall twice as much as Precision, it keeps increasing as the threshold drops, even at the $99\%$ target. In practice, the false-alarm rate should be read alongside $F_2$ rather than relying on $F_2$ alone.
* **Practical Takeaway:** A $90\%$ to $95\%$ validation target is a reasonable middle ground for a recall-focused alarm. A natural extension is to set the threshold **per engine** (e.g., relative to each engine's own early healthy flights), since much of the mismatch comes from engine-to-engine differences in baseline score levels.

## 16.0 Summary for the Report

* **Baseline:** Raw sensor means gave weak separation (validation ROC-AUC ≈ 0.57), and performance depended strongly on flight class.
* **Diagnosis:** Operating conditions explain $>99.9\%$ of each sensor's variation, masking the much smaller degradation signal (`T48`: $\approx 5\degree C$ degradation shift vs. $\approx 10\degree C$ of flight-to-flight variation).
* **Fix:** Replace each sensor with its residual from a model of healthy behavior given flight conditions, fit on healthy training data only.
* **Additional Gains:** 5-flight rolling median (gradual degradation) and virtual sensors (simulator-derived; reported with a caveat).
* **Downsampling Check:** Features from the full 1 Hz data performed the same as the downsampled data, so downsampling is justified.
* **Final Model (Selected on Validation):**
    * Validation: $\text{ROC-AUC } 0.92, \text{PR-AUC } 0.97$
    * Test: $\text{ROC-AUC } 0.83, \text{PR-AUC } 0.93$ (single seed)
    * *(See Notebook #3 for the seed-averaged result and hyperparameter sensitivity)*
* **Operating-Condition Check:** Correlation between healthy-flight scores and flight conditions dropped from $|r| \approx 0.31$–$0.60$ (baseline) to $\approx 0.25$–$0.33$ (final): reduced, but not eliminated.
* **At a Fixed Alarm Threshold (90% Recall Target, Chosen on Validation):** Test Recall $0.69$, False-Alarm Rate $0.015$, Precision $0.99$, $F_2 \; 0.74$; every test engine flagged $1$ to $11$ flights after onset.
* **Behavior:** Every held-out engine scores well above chance across all flight classes; scores stay flat while healthy & rise with degradation progress.
* **Main Limitation:** Scores rise gradually after the healthy-to-unhealthy transition, so the earliest unhealthy flights are the hardest to detect. A single alarm threshold also does not transfer perfectly between engines.